# 01 — CholecSeg8k Exploratory Data Analysis

Inspect the CholecSeg8k segmentation dataset, the 13 -> 6 class remapping, and
the per-class pixel distribution.

**Prerequisite:** download the dataset first — `bash scripts/download_cholecseg8k.sh`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.data.cholecseg8k import (
    CLASS_NAMES, CHOLECSEG8K_COLOR_MAP, load_cholecseg8k, remap_mask,
)
print("6-class schema:", CLASS_NAMES)

## 1. Load the dataset

In [ ]:
ds = load_cholecseg8k()
print(ds)

# A DatasetDict unwraps to a single split.
split = ds["train"] if hasattr(ds, "keys") else ds
print("columns:", split.column_names)
print("n examples:", len(split))

## 2. Sample frame, raw mask, and remapped 6-class mask

In [ ]:
record = split[0]
# TODO: adjust the column names below to the real HF schema if they differ.
image = np.asarray(record["image"].convert("RGB"))
raw_mask = np.asarray(record["color_mask"])
mask6 = remap_mask(raw_mask)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(image);    ax[0].set_title("frame")
ax[1].imshow(raw_mask); ax[1].set_title("raw CholecSeg8k mask")
ax[2].imshow(mask6, vmin=0, vmax=5, cmap="tab10")
ax[2].set_title("remapped 6-class mask")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 3. Verify the mask encoding  (IMPORTANT)

`CHOLECSEG8K_COLOR_MAP` in `src/data/cholecseg8k.py` is a best-effort palette
(the HuggingFace Hub was unreachable when the code was written). This cell
lists the values actually present in the masks — confirm they match, and
correct the table if they do not.

In [ ]:
unique = set()
for i in range(min(50, len(split))):
    m = np.asarray(split[i]["color_mask"])
    if m.ndim == 3:
        unique |= {tuple(int(c) for c in px[:3]) for px in m.reshape(-1, m.shape[-1])}
    else:
        unique |= set(int(v) for v in np.unique(m))

print(f"{len(unique)} unique mask values/colors in 50 masks:")
for u in sorted(unique):
    known = CHOLECSEG8K_COLOR_MAP.get(u) if isinstance(u, tuple) else None
    print("  ", u, "->", known or "UNKNOWN — update CHOLECSEG8K_COLOR_MAP")

## 4. Per-class pixel distribution (6-class schema)

In [ ]:
counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
for i in range(min(200, len(split))):
    m = remap_mask(np.asarray(split[i]["color_mask"]))
    counts += np.bincount(m.ravel(), minlength=len(CLASS_NAMES))[:len(CLASS_NAMES)]
frac = counts / counts.sum()

plt.figure(figsize=(8, 4))
plt.bar(CLASS_NAMES, frac)
plt.ylabel("pixel fraction")
plt.title("CholecSeg8k 6-class pixel distribution (subset)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

for name, f in zip(CLASS_NAMES, frac):
    print(f"  {name:14s} {f:8.4%}")

## Notes

- The distribution above is expected to be heavily imbalanced (liver dominant,
  cystic duct rare). Quantified and addressed in `02_class_imbalance_analysis`.
- `cystic_artery` should be ~0% here — CholecSeg8k does not label it.